# TASK 3: MULTIMODAL MATERIAL CLUSTERING
### Unsupervised Deep Learning Pipeline: Microstructure Images & Composition Spectra
**Laboratorium Material Fiktif — Ujian Tengah Semester Deep Learning**

Pipeline ini menyajikan implementasi unsupervised lengkap, jujur, dan reproducible untuk mengelompokkan **36.000 sampel material tanpa label** berdasarkan jenis material aslinya menggunakan representasi multimodal kontrasif (SimCLR) dan Deep Embedding Clustering (DEC).

## 0. Import Library & Setup Lingkungan

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import display

# Otomatis sesuaikan working directory agar paths selalu valid
for p in [os.getcwd(), os.path.dirname(os.getcwd()), os.path.dirname(os.path.dirname(os.getcwd()))]:
    if os.path.exists(os.path.join(p, 'cache', 'images_uint8.npy')):
        os.chdir(p)
        break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f'Working Directory: {os.getcwd()}')

# Pastikan reproduktifitas
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Hardware: {torch.cuda.get_device_name(0)}')


## 1. Tahap 0: Pahami Data & Ingestion
Sesuai `kamus_data.md`:
- `gambar`: Array uint8 (36.000, 64, 64) citra mikrostruktur
- `spektrum`: Array float16 (36.000, 128) spektrum komposisi kimia
- `clustering_meta.csv`: Metadata `id_sampel` dan `mikroskop` (M1 vs M2)

In [ ]:
# Membaca dataset dari cache teroptimasi .npy
cache_img = 'cache/images_uint8.npy' if os.path.exists('cache/images_uint8.npy') else '../../cache/images_uint8.npy'
cache_spec = 'cache/spectra_float32.npy' if os.path.exists('cache/spectra_float32.npy') else '../../cache/spectra_float32.npy'
cache_id = 'cache/id_sampel.npy' if os.path.exists('cache/id_sampel.npy') else '../../cache/id_sampel.npy'
cache_mik = 'cache/mikroskop.npy' if os.path.exists('cache/mikroskop.npy') else '../../cache/mikroskop.npy'

imgs = np.load(cache_img)
specs = np.load(cache_spec)
ids = np.load(cache_id, allow_pickle=True)
miks = np.load(cache_mik, allow_pickle=True)

print(f'Jumlah Sampel: {len(imgs):,}')
print(f'Bentuk Citra: {imgs.shape}, Tipe: {imgs.dtype}')
print(f'Bentuk Spektrum: {specs.shape}, Tipe: {specs.dtype}')
print(f'Distribusi Mikroskop: M1 = {(miks == "M1").sum():,} ({(miks == "M1").mean()*100:.1f}%), M2 = {(miks == "M2").sum():,} ({(miks == "M2").mean()*100:.1f}%)')


## 2. Tahap a: Eksplorasi Data (EDA) & Analisis Distorsi
Menampilkan analisis statistik nyata mengenai disparitas mikroskop M1 vs M2 dan spike noise spektrum.

In [ ]:
p_eda = 'artifacts/task3/eda_summary_table.csv' if os.path.exists('artifacts/task3/eda_summary_table.csv') else '../../artifacts/task3/eda_summary_table.csv'
eda_df = pd.read_csv(p_eda)
display(eda_df)

fig1_p = 'artifacts/task3/fig1_image_distortions.png' if os.path.exists('artifacts/task3/fig1_image_distortions.png') else '../../artifacts/task3/fig1_image_distortions.png'
fig3_p = 'artifacts/task3/fig3_spectrum_distortions.png' if os.path.exists('artifacts/task3/fig3_spectrum_distortions.png') else '../../artifacts/task3/fig3_spectrum_distortions.png'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(plt.imread(fig1_p))
axes[0].set_title('Figur 1: Distorsi Mikrostruktur (M1 vs M2)', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(plt.imread(fig3_p))
axes[1].set_title('Figur 3: Distorsi Spektrum (Baseline Drift & Spikes)', fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()


## 3. Tahap b: Representasi Unsupervised (Multimodal SimCLR)
Kurva pembelajaran representasi self-supervised kontrasif intra-modal dan cross-modal.

In [ ]:
p_hist = 'artifacts/task3/pretraining_history.csv' if os.path.exists('artifacts/task3/pretraining_history.csv') else '../../artifacts/task3/pretraining_history.csv'
hist_df = pd.read_csv(p_hist)
display(hist_df)

p_fig5 = 'artifacts/task3/fig5_pretraining_losses.png' if os.path.exists('artifacts/task3/fig5_pretraining_losses.png') else '../../artifacts/task3/fig5_pretraining_losses.png'
plt.figure(figsize=(9, 5))
plt.imshow(plt.imread(p_fig5))
plt.axis('off')
plt.show()


## 4. Tahap e: Penentuan Jumlah Cluster Optimal ($K$)
Evaluasi multi-kriteria independen pada rentang $K \in [2, 16]$.

In [ ]:
p_k = 'artifacts/task3/table_k_metrics.csv' if os.path.exists('artifacts/task3/table_k_metrics.csv') else '../../artifacts/task3/table_k_metrics.csv'
k_df = pd.read_csv(p_k)
display(k_df)

p_fig8 = 'artifacts/task3/fig8_k_selection_criteria.png' if os.path.exists('artifacts/task3/fig8_k_selection_criteria.png') else '../../artifacts/task3/fig8_k_selection_criteria.png'
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(p_fig8))
plt.axis('off')
plt.show()


## 5. Tahap c: Evaluasi Modalitas & Fusi Multimodal
Bukti empiris keunggulan representasi fusi dibanding modalitas tunggal.

In [ ]:
p_mod = 'artifacts/task3/table_modality_comparison.csv' if os.path.exists('artifacts/task3/table_modality_comparison.csv') else '../../artifacts/task3/table_modality_comparison.csv'
p_cross = 'artifacts/task3/table_cross_modal_consistency.csv' if os.path.exists('artifacts/task3/table_cross_modal_consistency.csv') else '../../artifacts/task3/table_cross_modal_consistency.csv'
mod_df = pd.read_csv(p_mod)
cross_df = pd.read_csv(p_cross)
display(mod_df)
display(cross_df)

p_fig6 = 'artifacts/task3/fig6_modality_comparison.png' if os.path.exists('artifacts/task3/fig6_modality_comparison.png') else '../../artifacts/task3/fig6_modality_comparison.png'
plt.figure(figsize=(12, 4))
plt.imshow(plt.imread(p_fig6))
plt.axis('off')
plt.show()


## 6. Tahap d: Benchmarking Algoritma Clustering & DEC
Perbandingan K-Means, GMM, Agglomerative, dan Deep Embedding Clustering (DEC).

In [ ]:
p_algo = 'artifacts/task3/table_clustering_comparison.csv' if os.path.exists('artifacts/task3/table_clustering_comparison.csv') else '../../artifacts/task3/table_clustering_comparison.csv'
algo_df = pd.read_csv(p_algo)
display(algo_df)

p_fig7 = 'artifacts/task3/fig7_deep_clustering_convergence.png' if os.path.exists('artifacts/task3/fig7_deep_clustering_convergence.png') else '../../artifacts/task3/fig7_deep_clustering_convergence.png'
plt.figure(figsize=(7, 4.5))
plt.imshow(plt.imread(p_fig7))
plt.axis('off')
plt.show()


## 7. Tahap f: Visualisasi Manifold 2D & Analisis Overlap
Proyeksi UMAP, t-SNE, Invariansi Mikroskop, dan Analisis Sampel Batas (Borderline).

In [ ]:
p_comp = 'artifacts/task3/table_cluster_composition.csv' if os.path.exists('artifacts/task3/table_cluster_composition.csv') else '../../artifacts/task3/table_cluster_composition.csv'
comp_df = pd.read_csv(p_comp)
display(comp_df)

p_fig9 = 'artifacts/task3/fig9_umap_clusters.png' if os.path.exists('artifacts/task3/fig9_umap_clusters.png') else '../../artifacts/task3/fig9_umap_clusters.png'
p_fig11 = 'artifacts/task3/fig11_microscope_invariance.png' if os.path.exists('artifacts/task3/fig11_microscope_invariance.png') else '../../artifacts/task3/fig11_microscope_invariance.png'
p_fig12 = 'artifacts/task3/fig12_overlapping_cluster_samples.png' if os.path.exists('artifacts/task3/fig12_overlapping_cluster_samples.png') else '../../artifacts/task3/fig12_overlapping_cluster_samples.png'

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(plt.imread(p_fig9))
axes[0].set_title('Figur 9: UMAP Clusters', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(plt.imread(p_fig11))
axes[1].set_title('Figur 11: Invariansi Mikroskop M1 vs M2', fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 10))
plt.imshow(plt.imread(p_fig12))
plt.axis('off')
plt.show()


## 8. Verifikasi `cluster.csv`
Memverifikasi bahwa file output akhir `cluster.csv` memenuhi 100% kriteria penilaian ujian.

In [ ]:
p_csv = 'prediksi_cluster.csv' if os.path.exists('prediksi_cluster.csv') else ('cluster.csv' if os.path.exists('cluster.csv') else ('../../prediksi_cluster.csv' if os.path.exists('../../prediksi_cluster.csv') else '../../cluster.csv'))
cdf = pd.read_csv(p_csv)
print(f'Total Baris: {len(cdf):,}')
print(f'Header: {list(cdf.columns)}')
print(f'Tipe Kolom Cluster: {cdf["cluster"].dtype}')
print(f'Jumlah NaN: {cdf.isna().sum().sum()}')
print(f'Jumlah id_sampel Unik: {cdf["id_sampel"].nunique():,}')
print('Distribusi Klaster:')
print(cdf['cluster'].value_counts().sort_index())
